# 层次分析模型--正互反矩阵最大特征根和特征向量的实用算法

### 1.幂法

 设 $A$ 为 $n \times n$ 正互反矩阵，步骤如下

$
a. 任取\,n\,维 非负归一化初始列向量(分量之和为1)w^{(0)} ~~~~~~~~~~~ \\
b. 迭代计算\,\tilde{w}^{(k + 1)}\, = Aw^{(k)}\,,~k =  0,1,2,... ~~~~~~~~~~~~~~~~~~ \\
c. \tilde{w}^{(k + 1)}归一化,即令\,w^{(k + 1)} = \frac{\tilde{w}^{(k + 1)}}{\sum_{i = 1}^{n}\tilde{w}^{(k + 1)}} ~~~~~~~~~~~~~~~~~~~ \\
 d. 收敛判断，即用一个预定的精度\,\varepsilon\, 当|w_{i}^{(k + 1)} - w_{i}^{(k)}| < \varepsilon \,时 w^{(k + 1)}为特征向量 ~ \\
 e. 计算最大特征根\lambda = \frac{1}{n}\sum_{i = 1}^{n}\frac{\tilde{w_i}^{(k + 1)}}{w_i^{(k}} ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
$


``` cpp



#include<iostream>
#include<vector>
#include<random>
#include<algorithm>
#include<ranges>
#include<print>

#define fun auto
#define let auto
#define in :

using namespace std::views;

template<typename T>
struct matrix;

template<typename T>
fun one(int n) -> matrix<T>
{
    matrix<T> ret(n, n, {});
    for(int i: iota(0, n)) {
        ret[i][i] = T{ 1 };
    }
    return ret;
}

template<typename T, std::integral I>
fun pow(const matrix<T> &v, I k) -> matrix<T>
{
    auto a{ one<T>(v.n) },tv{ v };
    for(; k; k >>= 1) {
        if(k & 1) {
            a = a * tv;
        }
        tv = tv * tv;
    }
    return a;
}

template<typename T>
struct matrix
{

    template<typename U>
    friend struct matrix;

    matrix(int n, int m) : a(new T[n * m]), n(n), m(m)
    {}

    matrix(int n, int m, const T &val) : matrix(n, m)
    { std::ranges::fill_n(a, n * m, val); }

    matrix(const matrix &v) : matrix(v.n,v.m)
    { std::ranges::copy_n(v.a, n * m, a); }

    template<typename U>
    requires std::convertible_to<U, T>
    matrix(const matrix<U> &v) : matrix(v.n,v.m)
    { std::ranges::copy_n(v.a, n * m, a); }

    matrix(matrix &&v) noexcept: a(v.a), n(v.n), m(v.m)
    { v.a = nullptr; }

    template<int N_, int M_>
    constexpr matrix(const T (&il)[N_][M_]) : matrix(N_,M_)
    {
        T *it = a;
        for(auto &v: il) {
            std::ranges::copy(v, it);
            it += M_;
        }
    }

    template<int I,int M_,typename U>
    fun constexpr make_matrix(U const (&il)[M_])
    {
        let d = I * M_;
        for(let i = 0; i != M_; ++i) {
            a[d + i] = il[i];
        }
    }

    template<int I,int M_,typename U,typename... R>
    fun constexpr make_matrix(U const (&il)[M_],R const (&...ils)[M_])
    {
        let d = I * M_;
        for(let i = 0; i != M_; ++i) {
            a[d + i] = il[i];
        }
        make_matrix<I + 1,M_,R...>(ils...);
    }

    template<typename U,typename... R,int M_>
    constexpr matrix(U const (&il)[M_],R const (&...ils)[M_]) : matrix(sizeof...(ils) + 1,M_)
    {
        for(let i = 0; i != M_; ++i) {
            a[i] = il[i];
        }
        if constexpr (sizeof...(ils)) {
            make_matrix<1, M_, R...>(ils...);
        }
    }

    template<typename U>
    fun operator=(const matrix<U> &v) -> matrix &
    {
        n = v.n;
        m = v.m;
        T *tmp = new T[n * m];
        std::ranges::copy_n(v.a, n * m, tmp);
        a = tmp;
        return *this;
    }

    fun operator=(matrix &&v) noexcept -> matrix &
    {
        n = v.n;
        m = v.m;
        a = v.a;
        v.a = nullptr;
        return *this;
    }

    fun operator[](int i) noexcept -> T *
    { return a + (i * m); }

    fun operator[](int i) const noexcept -> const T *
    { return a + (i * m); }

    fun operator^(std::integral auto p) -> matrix<T>
    { return pow(*this, p); }

    fun operator*=(const T &val)
    { std::ranges::transform(a, a + n * m, a, [&val](const T &v) -> T { return v * val; }); }

    fun operator*=(matrix const& v)
    { return *this = *this * v; }

    fun operator%=(const T &val)
    { std::ranges::transform(a, a + n * m, a, [&val](const T &v) -> T { return v % val; }); }

    fun operator>>=(const matrix& v) -> matrix&
    { return *this = v * *this; }

    fun operator<<=(const matrix& v) -> matrix&
    { return *this = *this * v; }


    fun friend operator>>(std::istream& is,matrix& v) -> std::istream&
    { std::copy_n(std::istream_iterator<T>{ is },v.n * v.m,v.a); return is; }

    fun friend operator<<(std::ostream& os,const matrix& v) -> std::ostream&
    {
        T* it = v.a;
        for(int i : iota(0,v.n)) {
            std::ranges::copy_n(it,v.m,std::ostream_iterator<T>{ std::cout," " });
            std::cout << '\n';
            it += v.m;
        }
        return os;
    }

    template<std::invocable<T &, T &> Op>
    fun apply(int i, int j, Op op) -> void
    {
        let x = (*this)[i], y = (*this)[j];
        for(let _ in std::views::iota(0, m)) {
            op(*x++, *y++);
        }
    }

    template<std::invocable<T &, T &> Op>
    fun tapply(int i, int j, Op op) -> void
    {
        let x = a + i, y = a + j;
        for(let _ in std::views::iota(0, n)) {
            op(*x, *y);
            x += m, y += m;
        }
    }

    fun solve() -> std::tuple<bool,int,int>
    {

        let constexpr NOTE =
                "返回是否有解,rank,使用方程个数"
                "如果无解，第三个参数无意义"
        ;
        let constexpr eps = 1e-6;
        let dr = 0,cnt = 0;
        for(let i in std::views::iota(0, std::min(n,m - 1))) {
            let ir = std::views::iota(i, n);
            let it = *std::ranges::find_if(ir, [](auto const& v) { return bool(v); }, [&](auto k) { return (*this)[k][i]; });
            while(it == n and i + dr + 2 < m) {
                tapply(i, m - 1 - ++dr, std::ranges::swap);
                it = *std::ranges::find_if(ir, [](auto const& v) { return bool(v); }, [&](auto k) { return (*this)[k][i]; });
            }
            cnt = std::max(cnt,it);
            if(it == n) {
                return { std::ranges::any_of(ir, [](auto const& v) {
                    if constexpr(std::floating_point<T>) {
                        return std::abs(v) >= eps;
                    } else {
                        return bool(v);
                    }
                },[&](auto i) { return (*this)[i][m - 1]; }),i + 1,cnt + 1 };
            }
            if(it != i) {
                apply(it, i, std::ranges::swap);
            }
            if(std::abs((*this)[i][i] - 1) >= eps) {
                for(let j in std::views::iota(i, m) | std::views::reverse) {
                    (*this)[i][j] /= (*this)[i][i];
                }
            }
            for(let k in iota(0, n) | filter([&](auto k) { return k != i; })) {
                let const &coe = (*this)[k][i];
                apply(i, k, [coe](T &lhs, T &rhs) {
                    rhs -= lhs * coe;
                });
            }
        }
        return { true,m - 1,cnt + 1 };
    }

    ~matrix()
    { delete[] a; }

    template<typename U1, typename U2>
    fun friend operator*(const matrix<U1> &x, const matrix<U2> &y) -> matrix<std::common_type_t<U1, U2>>;

    int n, m;
private:
    T *a;
public:

    fun one() -> matrix
    { return one<T>(n); }

};

template<typename U, typename... R, int M_>
matrix(U const (&il)[M_], R const (&...ils)[M_]) -> matrix<std::common_type_t<U, R...>>;

template<typename U1, typename U2>
fun operator*(const matrix<U1> &x, const matrix<U2> &y) -> matrix<std::common_type_t<U1, U2>>
{
    matrix<std::common_type_t<U1, U2>> v(x.n, y.m, {});
    for(int i: iota(0, x.n)) {
        for(int c: iota(0, x.m)) {
            for(int j: iota(0, y.m)) {
                v[i][j] += x[i][c] * y[c][j];
            }
        }
    }
    return v;
}




template<typename Tn,typename Td = Tn>
requires (std::integral<Tn> or std::same_as<Tn,__int128>) and (std::integral<Td> or std::same_as<Td,__int128>)
struct frac
{

    constexpr frac(Tn num,Td den) noexcept : num(num),den(den)
    {
        if(den < 0)
        {
            this->den = -den;
            this->num = -num;
        }
    }

    constexpr frac(Tn num) noexcept : num(num),den(1){}

    constexpr frac() noexcept requires std::default_initializable<Tn> and std::default_initializable<Td> = default;

    explicit constexpr operator double() const noexcept
    {
        return static_cast<double>(num) / den;
    }

    explicit constexpr operator bool() const noexcept
    { return num; }

    [[nodiscard]]
    fun constexpr floor() const noexcept -> std::common_type_t<Tn,Td>
    {
        return num % den ? (num / den) - (num > 0 != den > 0) : num / den;
    }

    [[nodiscard]]
    fun constexpr ceil() const noexcept -> std::common_type_t<Tn,Td>
    {
        return num % den ? (num / den) + (num > 0 == den > 0) : num / den;
    }

    fun constexpr reduce() noexcept -> frac&
    {
        Tn g{ std::gcd(num,den) };
        num /= g;
        den /= g;
        return *this;
    }

    fun constexpr operator+=(const frac& n) noexcept -> frac&
    {
        num = num * n.den + n.num * den;
        den *= n.den;
        return *this;
    }

    fun constexpr operator-=(const frac& n) noexcept -> frac&
    {
        num = num * n.den - n.num * den;
        den *= n.den;
        return *this;
    }

    fun constexpr operator*=(const frac& n) noexcept -> frac&
    {
        num *= n.num;
        den *= n.den;
        return *this;
    }

    fun constexpr operator/=(const frac& n) noexcept -> frac&
    {
        num *= n.den;
        den *= n.num;
        if(den < 0)
        {
            num = -num;
            den = -den;
        }
        return *this;
    }

    fun constexpr friend operator+(frac x,const frac& y) noexcept -> frac
    {
        x += y;
        return x;
    }

    fun constexpr friend operator-(frac x,const frac& y) noexcept -> frac
    {
        x -= y;
        return x;
    }

    fun constexpr friend operator*(frac x,const frac& y) noexcept  -> frac
    {
        x *= y;
        return x;
    }

    fun constexpr friend operator/(frac x,const frac& y) noexcept  -> frac
    {
        x /= y;
        return x;
    }

    fun constexpr friend operator-(frac x) noexcept -> frac
    {
        x.num = -x.num;
        return x;
    }

    fun constexpr friend operator==(const frac& x,const frac& y) noexcept -> bool
    {
        return x.num * y.den == y.num * x.den;
    }

    fun constexpr friend operator<=>(const frac& x,const frac& y) noexcept -> std::strong_ordering
    {
        return x.num * y.den <=> y.num * x.den;
    }

    fun friend operator<<(std::ostream& os,const frac& n) noexcept -> std::ostream&
    {
        Tn g{ std::gcd(n.num,n.den) };
        if(n.den == g)
        {
            return os << n.num / g;
        }
        else
        {
            return os << n.num / g << '/' << n.den / g;
        }
    }

    Tn num{};
    Td den{ 1 };

};

template<typename T>
requires std::is_integral_v<T>
frac(T) -> frac<T>;

namespace std
{
    template<typename Tn,typename Td>
    auto abs(frac<Tn,Td> const& f) -> frac<Tn,Td>
    {
        if(f < 0) {
            return -f;
        }
        return f;
    }
}

fun main() -> int
{
    using f = long double;
    let A = matrix {
        { f(1),f(2),f(6) },
        { f(1. / 2),f(1),f(4) },
        { f(1. / 6),f(1. / 4),f(1) },
    };

    let _ = std::mt19937{ std::random_device{}() };
    let r = std::uniform_int_distribution(1,10);
    let vectorize = [](auto& vec) { return iota(0,3) | transform([&](auto i) -> auto& { return vec[i][0]; }); };
    let normalization = [&](auto& vec){
        let sum = std::ranges::fold_left(vectorize(vec),0.,std::plus{});
        std::ranges::for_each(vectorize(vec),[&](auto& val){ val /= sum; });
    };
//    let w = matrix {{
//        {f(r(_)) },
//        { f(r(_)) },
//        { f(r(_)) },
//    }};
    let w = matrix {{
        { A[0][0] },
        { A[0][1] },
        { A[0][2] },
    }};

    let sum = std::ranges::fold_left(vectorize(w),f{},std::plus{});

    normalization(w);

    let last = matrix(1,3,f{});

    let constexpr eps = 1e-6L;
    while(true) {
        last = std::move(w);
        w = A * last;
        normalization(w);
        if(std::ranges::all_of(zip(vectorize(w),vectorize(last)),[&](auto tuple){ let const& [d1,d2] = tuple; return std::abs(d1 - d2) <= eps; })) {
            break;
        }
    }

    let lam = std::ranges::fold_left(zip(vectorize(w),vectorize(last)) | transform([&](auto const& tp) -> f {
        let const& [f1,f2] = tp;
        return f1 / f2;
    }),f{},std::plus{});

    std::println("lambda = {:.4f}"  "\n"  "特征向量 = [{:.4f} {:.4f} {:.4f}]",
               lam,w[0][0],w[1][0],w[2][0]
    );


    return 0;
}

```

### 2.和法

$a. 将A的每一列向量归一化$
<br><br>
$b. 按行求和得到一向量 w$
<br><br>
$c. 将w归一化，为近似特征向量$
<br><br>
$d. 计算 \lambda = \frac{1}{n}\sum_{i = 1}^n \frac{Aw_i}{w_i}$

### 3.根法

$a. 按列归一化$
<br><br>
$b. 按行求积并开n次方根$
<br><br>
$c. 归一化求\lambda$